# LongFlow — Gate Night 7 (capture prerequisites)

Runtime: **L4 GPU**. ~45–60 min, ~$2–3. Two arms, five renders; artifacts
mirror to Drive per run; reruns skip completed work. Pre-registered criteria:
`experiments/p1_flow_head/NOTES.md` (Gate Night 7 entry).

| cell | arm | decides |
|---|---|---|
| 4 | **T: teacher under feedback noise** — stock DDPM head, σ = 0 / 0.2 / 0.3 | is the capture-v2 design viable? (noised-teacher outputs as training targets) |
| 6 | **A: σ=0.4 anomaly reroll** — 20K head, euler4, seeds 1 and 2 | variance vs dead zone |
| 8 | FD backfill for all five | |
| 10 | Bundle | |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "GN7 v1.0 (2026-08-14): teacher-under-noise + sigma-0.4 reroll"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
import numpy as np
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow or drag bundle"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import euler_sample
from src.flow_head.integration import FlowHeadPatch
from src.flow_head.trainer import load_checkpoint

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
OUT = "/content/gate_night7"
DRIVE_OUT = "/content/drive/MyDrive/longflow_gate7"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")
head20 = head20.to("cuda")

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def save_wav(tag, wav, extra=None):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    row = {"tag": tag, "audio_s": round(len(wav)/24000, 1)}
    if extra: row.update(extra)
    report["runs"].append(row)
    json.dump(report, open(f"{DRIVE_OUT}/gate_night7_report.json", "w"), indent=2)
    print(row, flush=True)

def done(tag):
    if os.path.exists(f"{DRIVE_OUT}/{tag}.wav"):
        print(f"skip {tag}", flush=True)
        return True
    return False

report = {"runs": [], "notebook_version": NOTEBOOK_VERSION}
print(f"READY — {NOTEBOOK_VERSION}")


In [ ]:
# shared inputs — same construction as GN5/GN6 (identical script)
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern} — check Drive mount/paths")

sents = []
for f in drive_glob(f"{TRAIN_CACHE_DIR}/*.pt")[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
prompts = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")
P0 = prompts[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS = []
w = 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800: break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("script:", w, "words")
report["sweep_words"] = w
report["sweep_script"] = ABL_SCRIPT


## 4. Arm T — teacher under feedback noise (blocks capture v2)

Stock DDPM head (NO FlowHeadPatch). Noise on the acoustic connector output,
but only for **per-frame generation calls** — the prompt encoding processes
the whole voice sample in one call with sequence length > 1, generation
feeds back one frame at a time (seq len 1). The hook gates on that and logs
the first few call shapes so the scorer can verify the gate held (the GN5
v1 lesson: an ungated hook erases the speaker). ~7–10 min per render
(teacher is ~1.5× slower than the head).


In [ ]:
class TeacherNoise:
    """sigma-noise on per-frame feedback only; prompt (seq>1) untouched.
    Logs the first call shapes to verify the gating assumption."""
    def __init__(self, module, sigma):
        self.module, self.sigma = module, sigma
        self.mu, self.var = None, None
        self.h = None
        self.shapes = []          # first calls, for the gate audit
        self.noised = self.skipped = 0
    def __enter__(self):
        def hook(mod, args, out):
            t = out[0] if isinstance(out, tuple) else out
            if len(self.shapes) < 5:
                self.shapes.append(list(t.shape))
            seq_len = t.shape[1] if t.ndim >= 2 else 1
            if seq_len != 1:      # prompt / multi-frame encoding: hands off
                self.skipped += 1
                return out
            with torch.no_grad():
                if self.mu is None:
                    self.mu = t.mean().detach(); self.var = t.var().detach()
                else:
                    self.mu = 0.99*self.mu + 0.01*t.mean().detach()
                    self.var = 0.99*self.var + 0.01*t.var().detach()
                if self.sigma <= 0:
                    return out
                self.noised += 1
                new = t + torch.randn_like(t) * (self.sigma * self.var.sqrt())
            return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
        self.h = self.module.register_forward_hook(hook)
        return self
    def __exit__(self, *exc):
        if self.h: self.h.remove()

for sigma, tag in [(0.0, "t7_sig000"), (0.2, "t7_sig020"), (0.3, "t7_sig030")]:
    if done(tag):
        continue
    torch.manual_seed(0)
    with TeacherNoise(model.model.acoustic_connector, sigma) as tn, \
         torch.inference_mode():
        out = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    save_wav(tag, wav, {"sigma": sigma, "hook_shapes": tn.shapes,
                        "noised_calls": tn.noised, "skipped_calls": tn.skipped})
    assert tn.skipped >= 1, "gate audit: expected at least one multi-frame (prompt) call skipped"


## 6. Arm A — the σ=0.4 anomaly, rerolled

GN6's σ=0.4 died between two survivors at seed 0. Two fresh seeds, 20K head,
euler4 — otherwise byte-identical to `g6_sig040`. ~3 min per render.


In [ ]:
class NoiseIntervention:
    def __init__(self, module, sigma_fn, active_fn=None):
        self.module, self.sigma_fn = module, sigma_fn
        self.active_fn = active_fn or (lambda: True)
        self.calls_fn = lambda: 0
        self.mu, self.var = None, None
        self.h = None
    def __enter__(self):
        def hook(mod, args, out):
            if not self.active_fn():
                return out
            t = out[0] if isinstance(out, tuple) else out
            with torch.no_grad():
                if self.mu is None:
                    self.mu = t.mean().detach(); self.var = t.var().detach()
                else:
                    self.mu = 0.99*self.mu + 0.01*t.mean().detach()
                    self.var = 0.99*self.var + 0.01*t.var().detach()
                sigma = self.sigma_fn(self.calls_fn())
                if sigma <= 0:
                    return out
                new = t + torch.randn_like(t) * (sigma * self.var.sqrt())
            return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
        self.h = self.module.register_forward_hook(hook)
        return self
    def __exit__(self, *exc):
        if self.h: self.h.remove()

for seed in (1, 2):
    tag = f"a7_sig040_s{seed}"
    if done(tag):
        continue
    torch.manual_seed(seed)
    with FlowHeadPatch(model, head20, mean20, std20, nfe=4, sway=0.0,
                       sampler=euler_sample) as patch, \
         NoiseIntervention(model.model.acoustic_connector, lambda c: 0.4,
                           active_fn=lambda: patch.calls > 0) as nz, \
         torch.inference_mode():
        nz.calls_fn = lambda: patch.calls
        out = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    save_wav(tag, wav, {"sigma": 0.4, "seed": seed, "frames": patch.calls})


## 8. FD backfill — all five renders vs the GN3 teacher reference


In [ ]:
D_LAT = std20.numel()

def unwrap(z):
    while isinstance(z, (tuple, list)):
        z = z[0]
    if hasattr(z, "sample"):
        z = z.sample() if callable(z.sample) else z.sample
    while isinstance(z, (tuple, list)):
        z = z[0]
    assert torch.is_tensor(z), f"could not unwrap encoder output: {type(z)}"
    return z

def encode_latents(path):
    x, sr = sf.read(path, dtype="float32")
    z_all = []
    step = 24000 * 30
    with torch.inference_mode():
        for i in range(0, len(x), step):
            seg = torch.from_numpy(x[i:i+step])[None, None].to("cuda", torch.bfloat16)
            z = unwrap(model.model.acoustic_tokenizer.encode(seg))
            while z.ndim > 2:
                z = z.squeeze(0)
            if z.shape[-1] != D_LAT and z.shape[0] == D_LAT:
                z = z.T
            assert z.shape[-1] == D_LAT, f"latent width {z.shape} vs ckpt {D_LAT}"
            z_all.append(z.float().cpu())
    out = torch.cat(z_all)
    print(f"  {os.path.basename(path)}: {len(out)} frames ({len(out)/7.5:.0f}s)", flush=True)
    return out

def fd_curve(z, ref_mu, ref_cov, win=75):
    import scipy.linalg
    out = []
    for i in range(0, len(z) - win, win):
        w = z[i:i+win].numpy()
        mu, cov = w.mean(0), np.cov(w.T)
        d = mu - ref_mu
        covmean = scipy.linalg.sqrtm(cov @ ref_cov)
        if np.iscomplexobj(covmean): covmean = covmean.real
        out.append(float(d @ d + np.trace(cov + ref_cov - 2*covmean)))
    return out

teacher_z = encode_latents(f"{GATE3_DIR}/t1_turnsplit_p0.wav")
half = len(teacher_z) // 2
ref_mu, ref_cov = teacher_z[:half].numpy().mean(0), np.cov(teacher_z[:half].numpy().T)

TAGS = ["t7_sig000", "t7_sig020", "t7_sig030", "a7_sig040_s1", "a7_sig040_s2"]
fd = {"teacher_self": fd_curve(teacher_z[half:], ref_mu, ref_cov)}
for tag in TAGS:
    p = f"{DRIVE_OUT}/{tag}.wav"
    if os.path.exists(p):
        fd[tag] = fd_curve(encode_latents(p), ref_mu, ref_cov)
report["fd_curves"] = fd
json.dump(report, open(f"{DRIVE_OUT}/gate_night7_report.json", "w"), indent=2)
for k, v in fd.items():
    print(f"{k}: first={v[0]:.1f} med={sorted(v)[len(v)//2]:.1f} last={v[-1]:.1f} n={len(v)}")


## 10. Bundle

`gate_night7_bundle.zip` (zips from the Drive mirror). Mac:
`unzip -o ~/Downloads/gate_night7_bundle.zip -d experiments/p1_flow_head/audio/gate_night7`
then `.venv/bin/python experiments/p1_flow_head/score_gate_night7.py`.


In [ ]:
import zipfile
json.dump(report, open(f"{DRIVE_OUT}/gate_night7_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night7_bundle.zip", "w") as z:
    for f in glob.glob(f"{DRIVE_OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night7_bundle.zip")
